In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress

## Load MIMIC-IV Tables Necessary for Project

In [ ]:
# Load ICU Stays table
icu_stays = pd.read_csv('raw_data/icustays.csv.gz')
#icu_stays.head()

In [ ]:
# Load Admissions table
admissions = pd.read_csv('raw_data/admissions.csv.gz')
#admissions.head()

In [ ]:
# Load Patients table
patients = pd.read_csv('raw_data/patients.csv.gz')
#patients.head()

In [ ]:
# Load Item IDs table to map vitals data to the correct names
item_ids = pd.read_csv('raw_data/d_items.csv.gz')
#item_ids.head()

## Identify specific vital signs to be used for this project

In [ ]:
# Make note of vital sign item ids
# (heart rate, blood pressure, respiratory rate, temperature, and oxygen saturation)
pd.set_option('display.max_rows', None)
item_ids.loc[item_ids['label'].str.contains('heart rate|bp|respiratory rate|temperature|o2', case=False), 
                                            ['itemid', 'label']].drop_duplicates()

## Create a separate dataframe with vitals data for 5 major vital signs only

#### -------------------------------------------------------------------------
#### AI USAGE CITATION
#### Tool: Gemini
#### Prompt: "Write python code that creates a new filtered dataframe with only data 
#### on 5 vital signs from a csv.gz file with many vital signs. The file is big, so 
#### I want to ensure I'm proceeding efficiently. I have a vitals_map with the vital 
#### names and their respective IDs from a different file."
#### Usage: Used chunk iterator structure; used chunking loop approach.
#### -------------------------------------------------------------------------

In [ ]:
# Map itemids for 5 major vital signs
vitals_map = {
    220045: 'heart_rate',
    220179: 'sys_bp',
    220210: 'resp_rate',
    223761: 'temp_f',
    220277: 'spo2'
}

# Extract item ids only from mapping
target_itemids = list(vitals_map.keys())
# Create placeholder list for filtered chunks
filtered_chunks = []

# Read in chunks of 100,000 rows at a time, only pulling in the necessary columns from chartevents file to save memory
chunk_iterator = pd.read_csv(
    'raw_data/chartevents.csv.gz', 
    chunksize=100000, 
    usecols=['stay_id', 'charttime', 'itemid', 'valuenum']
)

for i, chunk in enumerate(chunk_iterator):
    # Only keep rows that match the 5 vital signs indicated above
    vitals_chunk = chunk[chunk['itemid'].isin(target_itemids)].copy()
    # Create clean names for each vital sign
    vitals_chunk['vital_name'] = vitals_chunk['itemid'].map(vitals_map)
    # Append the chunks to filtered chunks list
    filtered_chunks.append(vitals_chunk)

# Combine filtered rows into one dataframe
vitals = pd.concat(filtered_chunks, ignore_index=True)

In [ ]:
# View first 10 observations of new vitals dataframe
vitals.head(10)

## Merge All Necessary Tables into One Dataframe

#### Filter ICU Stays table to only MICU stays (for scope of project)

In [ ]:
micu_stays = icu_stays[icu_stays['first_careunit'] == 'Medical Intensive Care Unit (MICU)']
#micu_stays.head()

#### Merge patients, admissions, and vitals data to MICU stays

In [ ]:
micu_first_merge = micu_stays.merge(patients, on='subject_id', how='left')
micu_first_merge = micu_first_merge.merge(admissions, on='hadm_id', how='left')

# Pivot the vitals table so each vital has its own column
vitals_pivoted = vitals.pivot_table(
    index=['stay_id', 'charttime'],
    columns='vital_name',
    values='valuenum',
    aggfunc='mean' # Added in case a vital was logged twice in the same minute
).reset_index()

# Merge pivoted vitals data to main ICU dataframe
micu_merged = micu_first_merge.merge(vitals_pivoted, on='stay_id', how='left')
# Keep only the necessary columns for analysis
micu_merged = micu_merged[['stay_id', 'intime', 'outtime', 'los', 'admission_type', 'admission_location', 'race', 
                           'gender', 'anchor_age', 'charttime', 'heart_rate', 'resp_rate', 'spo2', 'sys_bp', 'temp_f']]

# Rename age column
micu_merged.rename(columns={'anchor_age': 'age'}, inplace=True)

micu_merged.head()

In [ ]:
# Nearly 3 million rows present
micu_merged.shape

In [ ]:
# View data types for any transformations needed
micu_merged.dtypes

## Initial Data Visualizations

### Create dataframe with only unique stays for exploratory visualizations

In [ ]:
unique_stays = micu_merged.drop_duplicates(subset=['stay_id'], keep='first')
unique_stays.head()

### Distribution of Length of Stay Variable - Histogram

In [ ]:
unique_stays['los'].hist(bins=25, figsize=(12,5))

plt.title("Length of Stay Distribution")
plt.xlabel("Days")

# Data is heavily right-skewed and hard to visualize in a standard histogram

### Distribution of Length of Stay Variable - Log Histogram

In [ ]:
unique_stays['los'].hist(bins=25, figsize=(12,5), log=True)

plt.title("Length of Stay Distribution (Log-Scaled Frequency)")
plt.xlabel("Days")

### Distribution of Length of Stay Variable - Boxplot (with extreme values removed to view trends more easily)

In [ ]:
plt.boxplot(unique_stays['los'].dropna(), showfliers=False)

plt.title("Length of Stay Distribution")
plt.ylabel("Days")

### Distribution of Age - Histogram

In [ ]:
plt.hist(unique_stays['age'], bins=15)

plt.title("Age Distribution")
plt.xlabel("Age")

### Distribution of Age - Boxplot

In [ ]:
unique_stays.boxplot(column='age', by='gender', grid=False)

plt.suptitle("")
plt.title("Age Distribution by Gender")
plt.xlabel("Gender: Female (0) / Male (1)")
plt.ylabel("Age")

### Plot Counts by Gender to Check for Imbalances

In [ ]:
unique_stays['gender'].value_counts().plot(kind='bar')

### Plot Admission Type Counts

In [ ]:
type_counts = unique_stays['admission_type'].value_counts().sort_values()

plt.barh(type_counts.index, type_counts.values)
plt.title("Unique Patient Counts by Admission Type")
plt.xlabel("Admission Type")
plt.ylabel("Number of Unique Patients")

### Plot Admission Location Counts

In [ ]:
location_counts = unique_stays['admission_location'].value_counts().sort_values()

plt.barh(location_counts.index, location_counts.values)
plt.title("Unique Patient Counts by Admission Location")
plt.xlabel("Admission Location")
plt.ylabel("Number of Unique Patients")

#### All Vitals Appear to Have Outliers Based on Distribution Charts

In [ ]:
# Plot distribution of each vital sign

vitals_list = ['heart_rate', 'sys_bp', 'resp_rate', 'spo2', 'temp_f']

micu_merged[vitals_list].hist(
    bins=20,
    figsize=(15, 10),
    layout=(2, 3),
    grid=False
)
plt.suptitle("Distribution of Vital Signs", fontsize=18)
plt.tight_layout()

## Data Cleaning

### Set Clinical Bounds

#### -------------------------------------------------------------------------
#### AI USAGE CITATION
#### Tool: Gemini
#### Prompt: "Write python code that loops through values and sets specific 
#### boundaries for each feature. Please recommend upper and lower clinical
#### bounds for each vital sign based on what is physiologically possible."
#### Usage: Used looping approach to clip high and low values; used counting 
#### approach to print the number of high and low values clipped for transparency;
#### used some of recommended high and low clinical bounds in conjunction with
#### other resources.
#### -------------------------------------------------------------------------

In [ ]:
# Define clinical bounds for vital signs by day
clinical_bounds_by_day = {
    'heart_rate': (30, 220),
    'sys_bp': (40, 280),
    'resp_rate': (4, 60),
    'spo2': (50, 100),
    'temp_f': (88, 110)
}

# Remove records with vitals outside clinical bounds in the by-day dataframe
for vital, (vmin, vmax) in clinical_bounds_by_day.items():
    if vital in micu_merged.columns:
        # Count how many values are outside the boundaries before clipping
        under_count = (micu_merged[vital] < vmin).sum()
        over_count = (micu_merged[vital] > vmax).sum()
        # Print counts
        if under_count > 0 or over_count > 0:
            print(f"{vital}: Removed {under_count} low values and {over_count} high values.")
        # Merge clipped vital values back into dataframe
        micu_merged[vital] = micu_merged[vital].clip(lower=vmin, upper=vmax)

### Confirm that the clinical bounds are set in by-day dataframe

In [ ]:
for vital in vitals_list:
    min = micu_merged[vital].min()
    max = micu_merged[vital].max()
    print(f"{vital}: Min = {min}, Max = {max}")

### Identify number of rows with missing length of stay variable

In [ ]:
micu_merged['los'].isna().sum()

### Clean dataframe

In [ ]:
# Drop rows where los is NA
micu_merged = micu_merged.dropna(subset=['los'])

# Convert time strings into pandas datetime objects
micu_merged['intime'] = pd.to_datetime(micu_merged['intime'])
micu_merged['outtime'] = pd.to_datetime(micu_merged['outtime'])
micu_merged['charttime'] = pd.to_datetime(micu_merged['charttime'])

## Feature Engineering

### Add column for day in ICU

In [ ]:
# Calculate time between admission and first chart record
time_delta = micu_merged['charttime'] - micu_merged['intime']

# Convert time into calendar days
micu_merged['icu_day'] = time_delta.dt.days + 1

# Drop any rows with data prior to ICU admission
micu_merged = micu_merged[micu_merged['icu_day'] >= 1]

### View descriptive statistics of numeric variables for unique stays

In [ ]:
unique_stays2 = micu_merged.drop_duplicates(subset=['stay_id'], keep='first')
unique_stays_numeric = unique_stays2[['los', 'age', 'heart_rate', 'resp_rate', 'spo2', 'sys_bp', 'temp_f']]
unique_stays_numeric.describe()

### Confirm use of 4 days as the extended stay threshold

In [ ]:
# Sort unique stays dataframe in order of length of stay
los_sorted = np.sort(unique_stays['los'])

# Create percentile scale for y-axis
y_values = np.arange(1, len(los_sorted) + 1) / len(los_sorted)

# Set plot size
plt.figure(figsize=(10, 6))

# Plot patients as dots to build a line graph
plt.plot(los_sorted, y_values * 100, marker='.', linestyle='none')

# Add lines to show 75th percentile falls on day 4
plt.axvline(x=4, color='red', linestyle='--', label="Day 4")
plt.axhline(y=75, color='green', linestyle='--', label="75th Percentile")

# Format plot
plt.title("Cumulative Distribution of ICU Length of Stay", fontsize=14, weight='bold')
plt.xlabel("Length of Stay (Days)", fontsize=12)
plt.ylabel("Percentage of Patients Included", fontsize=12)
plt.xlim(0, 30) # Looking only at the first 30 days for ease of viewing
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Create new column for extended stay threshold

In [ ]:
# Establish extended stay threshold and create new column
micu_merged['extended_stay'] = (micu_merged['los'] > 4).astype(int)

# Recreate unique stays df with new updates to micu_merged
unique_stays3 = micu_merged.drop_duplicates(subset=['stay_id'], keep='first')

# View distribution of target variable across unique stays
print(unique_stays3['extended_stay'].value_counts())

## Create By-Patient By-Day Dataframe

In [ ]:
# Create table with one row per stay id per day in ICU, using the daily average of vitals

micu_by_day = micu_merged.groupby(['stay_id', 'icu_day']).agg(
    los=('los', 'first'),
    extended_stay=('extended_stay', 'first'),
    age=('age', 'first'),
    gender=('gender', 'first'),
    race=('race', 'first'),
    daily_heart_rate=('heart_rate', 'mean'),
    daily_resp_rate=('resp_rate', 'mean'),
    daily_spo2=('spo2', 'mean'),
    daily_sys_bp=('sys_bp', 'mean'),
    daily_temp=('temp_f', 'mean'),
    admission_type=('admission_type', 'first'),
    admission_location=('admission_location', 'first'),
    in_time=('intime', 'first')
).reset_index()

micu_by_day.head()

### Confirm new number of patient day records

In [ ]:
micu_by_day.shape

### View descriptive statistics of numeric variables in by-day dataframe

In [ ]:
numeric_by_day = micu_by_day[['los', 'icu_day', 'age', 'daily_heart_rate', 'daily_resp_rate', 'daily_spo2', 'daily_sys_bp', 'daily_temp']]
numeric_by_day.describe()

### Handle Null Values in By-Patient By-Day Dataframe

In [ ]:
# Calculate the percentage of missing data per column in the by-day dataframe
na_summary_by_day = micu_by_day.isnull().mean() * 100
print(na_summary_by_day.sort_values(ascending=False))

### There are no days where no vital signs were recorded for a patient (all days have at least 1 vital record)

In [ ]:
vitals_all_na = micu_by_day[['daily_heart_rate', 'daily_resp_rate', 'daily_spo2', 'daily_sys_bp', 'daily_temp']].isna().all(axis=1)
vitals_all_na.sum()

### Check whether any patients are missing all records for a particular vital sign

In [ ]:
vitals_by_day = ['daily_heart_rate', 'daily_resp_rate', 'daily_spo2', 'daily_sys_bp', 'daily_temp']

for vital in vitals_by_day:
    all_na_per_stay = micu_by_day.groupby(['stay_id', 'icu_day'])[vital].apply(lambda x: x.isna().all())
    missing_stays_count = all_na_per_stay.sum()
    print(f"{vital}: {missing_stays_count} stays have completely missing data")

### Before imputing null values, create columns in by-day dataframe to note whether a value was originally missing from the data

In [ ]:
for vital in vitals_by_day:
    micu_by_day[f"{vital}_was_missing"] = micu_by_day[vital].isna().astype(int)

### Forward fill missing vital records in by-day dataframe

In [ ]:
# Sort by patient (stay id) and day to ensure correct sequential order
micu_by_day_clean = micu_by_day.sort_values(by=['stay_id', 'icu_day'])

# Forward fill null heart rate values
micu_by_day_clean['daily_heart_rate'] = micu_by_day_clean.groupby('stay_id')['daily_heart_rate'].ffill()
daily_heart_rate_nulls = micu_by_day_clean['daily_heart_rate'].isnull().sum()
print(f"Heart rate still contains {daily_heart_rate_nulls} null values")

# Forward fill null respiratory rate values
micu_by_day_clean['daily_resp_rate'] = micu_by_day_clean.groupby('stay_id')['daily_resp_rate'].ffill()
daily_resp_rate_nulls = micu_by_day_clean['daily_resp_rate'].isnull().sum()
print(f"Respiratory rate still contains {daily_resp_rate_nulls} null values")

# Forward fill null SpO2 values
micu_by_day_clean['daily_spo2'] = micu_by_day_clean.groupby('stay_id')['daily_spo2'].ffill()
daily_spo2_nulls = micu_by_day_clean['daily_spo2'].isnull().sum()
print(f"SpO2 still contains {daily_spo2_nulls} null values")

# Forward fill null blood pressure values
micu_by_day_clean['daily_sys_bp'] = micu_by_day_clean.groupby('stay_id')['daily_sys_bp'].ffill()
daily_sys_bp_nulls = micu_by_day_clean['daily_sys_bp'].isnull().sum()
print(f"Blood Pressure still contains {daily_sys_bp_nulls} null values")

# Forward fill null temperature values
micu_by_day_clean['daily_temp'] = micu_by_day_clean.groupby('stay_id')['daily_temp'].ffill()
daily_temp_nulls = micu_by_day_clean['daily_temp'].isnull().sum()
print(f"Temperature still contains {daily_temp_nulls} null values")

### Fill remaining NAs with the value of -1

In [ ]:
micu_by_day_clean[vitals_by_day] = micu_by_day_clean[vitals_by_day].fillna(-1)

### Confirm there are no longer null values in the by-day dataframe after forward filling + imputation

In [ ]:
na_summary_by_day_imputed = micu_by_day_clean.isnull().mean() * 100
print(na_summary_by_day_imputed.sort_values(ascending=False))

In [ ]:
# View first 10 observations of by-day dataframe with new columns added
micu_by_day_clean.head()

### Create dataframe with imputed -1 values excluded for visualization purposes

In [ ]:
plot_df_by_day = micu_by_day_clean.copy()
plot_df_by_day = plot_df_by_day[plot_df_by_day['icu_day'] >= 0]

for vital in vitals_by_day:
    plot_df_by_day[vital] = plot_df_by_day[vital].replace(-1, np.nan)

### View descriptive statistics of numeric variables in clean by-day dataframe

In [ ]:
numeric_by_day_clean = plot_df_by_day[['los', 'icu_day', 'age', 'daily_heart_rate', 'daily_resp_rate', 'daily_spo2', 'daily_sys_bp', 'daily_temp']]
numeric_by_day_clean.describe()

### Create Vitals Correlation Matrix for By-Day Dataframe

In [ ]:
# Using uncleaned data to avoid false correlations after forward filling
corr_df = micu_by_day.copy()

# Add columns for yesterday's vital averages and the day-over-day delta
for vital in vitals_by_day:
    yesterday = corr_df.groupby('stay_id')[vital].shift()
    corr_df[f'{vital}_delta'] = corr_df[vital] - yesterday

# Filter to only the vital sign data
corr_df = corr_df[['daily_heart_rate', 'daily_resp_rate', 'daily_spo2', 'daily_sys_bp', 
                              'daily_temp', 'daily_heart_rate_delta', 'daily_resp_rate_delta', 
                              'daily_spo2_delta', 'daily_sys_bp_delta', 'daily_temp_delta']]

corr_matrix = corr_df.corr()
corr_matrix

### Plot Correlation Matrix for By-Day Dataframe

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(corr_matrix)

## New Visualizations with Cleaned By-Day Data

### Plot distribution of each vital sign

In [ ]:
plot_df_by_day[vitals_by_day].hist(
    bins=20,
    figsize=(15, 10),
    layout=(2, 3),
    grid=False
)
plt.suptitle("Distribution of Vital Signs", fontsize=18)
plt.tight_layout()

### Plot 5 major vital signs over first 7 days in ICU - showing extreme values

In [ ]:
first_week1 = plot_df_by_day[plot_df_by_day['icu_day'] <= 7]

fig, axes = plt.subplots(5, 1, figsize=(12,18), sharex=True)

for i, vital in enumerate(vitals_by_day):
    sns.boxplot(
        data=first_week1,
        x='icu_day',
        y=vital,
        ax=axes[i]
    )
    axes[i].set_title(f"Distribution of {vital.replace('_', ' ').title()} Over First 7 Days", fontsize=12)
    axes[i].set_ylabel("")
    axes[i].set_xlabel("")
    axes[i].grid(axis='y', alpha=0.3)

axes[-1].set_xlabel("Day of ICU Stay", fontsize=12)

plt.show()

### Plot 5 major vital signs over first 7 days in ICU - removing extreme values

In [ ]:
first_week2 = plot_df_by_day[plot_df_by_day['icu_day'] <= 7]

fig, axes = plt.subplots(5, 1, figsize=(12,18), sharex=True)

for i, vital in enumerate(vitals_by_day):
    sns.boxplot(
        data=first_week2,
        x='icu_day',
        y=vital,
        ax=axes[i],
        showfliers=False
    )
    axes[i].set_title(f"Distribution of {vital.replace('_', ' ').title()} Over First 7 Days", fontsize=12, weight='bold', pad=10)
    axes[i].set_ylabel("")
    axes[i].set_xlabel("")
    axes[i].grid(axis='y', alpha=0.3)

axes[-1].set_xlabel("Day of ICU Stay", fontsize=12)

plt.show()

### View Temporal Trends for Each Vital Sign Over First 4 days

#### -------------------------------------------------------------------------
#### AI USAGE CITATION
#### Tool: Gemini
#### Prompt: "Write python code to plot a seaborn lineplot that shows temporal 
#### trends for metrics across a 4-day period, differentiating between patients 
#### who stayed less than 4 days and patients who stayed longer than 4 days."
#### Usage: Used target mapping approach; used seaborn lineplot structure and 
#### formatting approach with error bar; tuned formatting to better fit the 
#### dataset and be more easily reused for each vital sign. The looping code
#### was created by me, using by previous plotting code as a guide.
#### -------------------------------------------------------------------------

In [ ]:
# Isolate the first 4 days of ICU data to see early temporal trends (prior to extended stay window)
temporal_df = plot_df_by_day[plot_df_by_day['icu_day'] <= 4].copy()

# Map target
stay_lengths = temporal_df.groupby('stay_id')['los'].first()
extended_stay_map = (stay_lengths > 4).astype(int)
temporal_df['temp_target'] = temporal_df['stay_id'].map(extended_stay_map)

fig, axes = plt.subplots(5, 1, figsize=(12, 28), sharex=True)

# Create lineplot and loop through each vital sign
for i, vital in enumerate(vitals_by_day):
    sns.lineplot(
        data=temporal_df, 
        x='icu_day', 
        y=vital, 
        hue='temp_target',
        marker='o', 
        linewidth=2.5,
        errorbar=('ci', 95), 
        ax=axes[i],
        legend='brief' if i == 0 else False
    )
    # Clean up formatting
    axes[i].set_title(f"Temporal Trajectory of {vital.replace('_', ' ').title()} Over First 4 Days", fontsize=14, weight='bold', pad=10)
    axes[i].set_ylabel("")
    axes[i].set_xlabel("")
    axes[i].grid(axis='y', alpha=0.3)
    axes[i].set_xticks(range(1, 5))

# Custom legend labels
handles, labels = axes[0].get_legend_handles_labels()
axes[0].legend(handles, ["Normal Stay (≤4 Days)", "Extended Stay (>4 Days)"], title="Stay Length", loc='upper right', frameon=True)

plt.show()

## Create First 24 Hours Dataframe (Day 1)

### Initial setup of new dataframe

In [ ]:
# Ensure data is sorted chronologically so slope calculations are accurate
micu_hourly = micu_merged.sort_values(by=['stay_id', 'charttime'])

# Calculate hours elapsed since ICU admission
time_delta = micu_hourly['charttime'] - micu_hourly['intime']
micu_hourly['visit_hour'] = time_delta.dt.total_seconds() / 3600

# Isolate data to first 24 hours of stay only
first_24hours = micu_hourly[(micu_hourly['visit_hour'] >= 0) & (micu_hourly['visit_hour'] < 24)].copy()

### Calculate intra-day slopes of each vital sign

#### -------------------------------------------------------------------------
#### AI USAGE CITATION
#### Tool: Gemini
#### Prompt: "Write python code that calculates intraday slopes of vital signs."
#### Usage: Used approach to extract slope helper function.
#### -------------------------------------------------------------------------

In [ ]:
# Create function for generatng vital sign slopes
def extract_slope(patient_group, vital_column):
    # Remove NAs for calculations
    clean_data = patient_group[['visit_hour', vital_column]].dropna()
    # If there are not at least 2 datapoints to calculate a slope, show as missing
    if len(clean_data) < 2:
        return np.nan
    # Ignore all other values and return slope only
    slope, _, _, _, _ = linregress(clean_data['visit_hour'], clean_data[vital_column])
    return slope

#### -------------------------------------------------------------------------
#### AI USAGE CITATION
#### Tool: Gemini
#### Prompt: "Write python code that executes the slope extraction function
#### and adds columns with the calculated slopes to a new dataframe."
#### Usage: Used looping approach for executing extract slope function for 
#### each vital sign/patient & creation of missing flag.
#### -------------------------------------------------------------------------

### Aggregate necessary features and apply slope extraction function

In [ ]:
# Aggregate demographics and vital signs
micu_first_24hours = first_24hours.groupby('stay_id').agg(
    los=('los', 'first'),
    age=('age', 'first'),
    gender=('gender', 'first'),
    race=('race', 'first'),
    admission_type=('admission_type', 'first'),
    admission_location=('admission_location', 'first'),
    
    heart_rate_mean=('heart_rate', 'mean'),
    heart_rate_min=('heart_rate', 'min'),
    heart_rate_max=('heart_rate', 'max'),
    heart_rate_std=('heart_rate', 'std'),
    
    resp_rate_mean=('resp_rate', 'mean'),
    resp_rate_min=('resp_rate', 'min'),
    resp_rate_max=('resp_rate', 'max'),
    resp_rate_std=('resp_rate', 'std'),
    
    spo2_mean=('spo2', 'mean'),
    spo2_min=('spo2', 'min'),
    spo2_max=('spo2', 'max'),
    spo2_std=('spo2', 'std'),
    
    sys_bp_mean=('sys_bp', 'mean'),
    sys_bp_min=('sys_bp', 'min'),
    sys_bp_max=('sys_bp', 'max'),
    sys_bp_std=('sys_bp', 'std'),
    
    temp_mean=('temp_f', 'mean'),
    temp_min=('temp_f', 'min'),
    temp_max=('temp_f', 'max'),
    temp_std=('temp_f', 'std')
).reset_index()

# Mapping for variable names (temp variable name changed)
vitals_map_24hour = {
    'heart_rate': 'heart_rate',
    'resp_rate': 'resp_rate',
    'spo2': 'spo2',
    'sys_bp': 'sys_bp',
    'temp': 'temp_f'
}

# Implement extract_slope function for each vital sign / patient
for prefix, col_name in vitals_map_24hour.items():
    # Calculate slope per patient
    slope_series = first_24hours.groupby('stay_id').apply(
        lambda g: extract_slope(g, col_name),
        include_groups=False
    ).rename(f'{prefix}_slope')
    # Add flag for missing data
    flag_series = first_24hours.groupby('stay_id')[col_name].apply(
        lambda x: 1 if x.notna().sum() == 0 else 0
    ).rename(f'{prefix}_missing')
    # Merge slope and missing flag data back into first 24 hour dataframe
    micu_first_24hours = pd.merge(micu_first_24hours, slope_series, on='stay_id', how='left')
    micu_first_24hours = pd.merge(micu_first_24hours, flag_series, on='stay_id', how='left')

# Add extended stay column to first 24 hour dataframe (target variable)
micu_first_24hours['extended_stay'] = (micu_first_24hours['los'] > 4).astype(int)

### View 24-Hour dataframe head

In [ ]:
micu_first_24hours.head()

### Confirm new number of patient records

In [ ]:
micu_first_24hours.shape

### View descriptive statistics of numeric variables in the first 24-hour data

In [ ]:
numeric_first24 = micu_first_24hours.drop(columns=['stay_id', 'gender', 'race', 'admission_type', 'admission_location', 
                                                   'heart_rate_missing', 'resp_rate_missing', 'spo2_missing', 
                                                   'sys_bp_missing', 'temp_missing'])
numeric_first24.describe()

### Create Correlation Matrix for First 24-Hour Dataframe

In [ ]:
corr_matrix_first24 = numeric_first24.corr()
corr_matrix_first24

### Plot Correlation Matrix for First 24-Hour Dataframe

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(corr_matrix_first24)

### Review Presence of Null Values in First 24-Hours Dataframe (Day 1)

In [ ]:
# Calculate the percentage of missing data per column in the day 1 dataframe
na_summary_24hour = micu_first_24hours.isnull().mean() * 100
print(na_summary_24hour.sort_values(ascending=False))

#### All patients have at least 1 vital record in first 24 hours

In [ ]:
vitals_24hours = ['heart_rate_mean', 'resp_rate_mean', 'sys_bp_mean', 'sys_bp_mean', 'temp_mean']

vital_means_all_na_24hour = micu_first_24hours[vitals_24hours].isna().all(axis=1)
vital_means_all_na_24hour.sum()

## New Visualizations with Cleaned First-24-Hour Data

### Plot Each Vital Sign Against Age

In [ ]:
# Create subplots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Flatten axes for looping
axes_flat = axes.flatten()

# Loop through each of the 5 major vital signs
for i, vital in enumerate(vitals_24hours):
    sns.scatterplot(
        data=micu_first_24hours,
        x= 'age',
        y= vital,
        alpha=0.1,
        s=10,
        ax=axes_flat[i]
    )
    axes_flat[i].set_title(f"Age vs. {vital.replace('_', ' ').title()}")
    axes_flat[i].set_xlabel("Age")
    axes_flat[i].set_ylabel(vital.replace('_', ' ').title())

# Hide extra empty subplot
axes_flat.flat[-1].set_visible(False) 

### Plot Each Vital Sign Against Length of Stay

In [ ]:
# Create subplots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Flatten axes for looping
axes_flat = axes.flatten()

# Loop through each of the 5 major vital signs
for i, vital in enumerate(vitals_24hours):
    sns.scatterplot(
        data=micu_first_24hours,
        x= 'los',
        y= vital,
        alpha=0.1,
        s=10,
        ax=axes_flat[i]
    )
    axes_flat[i].set_title(f"Length of Stay vs. {vital.replace('_', ' ').title()}")
    axes_flat[i].set_xlabel("Length of Stay")
    axes_flat[i].set_ylabel(vital.replace('_', ' ').title())

# Hide extra empty subplot
axes_flat.flat[-1].set_visible(False) 

### Length of Stay vs. Age

In [ ]:
plt.scatter('los', 'age', data=micu_first_24hours, alpha=0.1, s=10)
plt.title("Length of Stay vs. Age")
plt.xlabel("Length of Stay")
plt.ylabel("Age")

## Export By-Day and First 24-Hour Dataframes for Model Creation

In [ ]:
micu_by_day_clean.to_pickle('micu_by_day.pkl')
micu_first_24hours.to_pickle('micu_24hours.pkl')